LASTTT

In [7]:
# ============================================================
# HYBRID RF-DBSCAN FIREWALL EXPERIMENT
# FINAL REVISION VERSION
# DATASET: RAW UNSW-NB15 + RAW CICIDS2017
# FEATURE REMOVED: flow_duration
# FINAL FEATURES: 8 packet-byte features
# ============================================================

import os
import re
import glob
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Sampling final sesuai revisi reviewer:
# Masing-masing dataset menyumbang 25.000 benign + 25.000 malicious
BENIGN_PER_DATASET = 25000
MALICIOUS_PER_DATASET = 25000

OUTPUT_DIR = Path("OUTPUT_RESULTS")
OUTPUT_DIR.mkdir(exist_ok=True)

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)

FEATURE_COLS = [
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
    "total_packets",
    "total_bytes",
    "packet_ratio",
    "byte_ratio"
]

print("Final features used:")
print(FEATURE_COLS)
print("\nREMOVED feature: flow_duration")

Final features used:
['fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes', 'total_packets', 'total_bytes', 'packet_ratio', 'byte_ratio']

REMOVED feature: flow_duration


In [8]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def find_column(df, candidates, target_name):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        f"Column for {target_name} not found. Available columns: {list(df.columns)[:80]}"
    )


def add_derived_features(df):
    df = df.copy()
    df["total_packets"] = df["fwd_packets"] + df["bwd_packets"]
    df["total_bytes"] = df["fwd_bytes"] + df["bwd_bytes"]
    df["packet_ratio"] = df["fwd_packets"] / (df["bwd_packets"] + 1)
    df["byte_ratio"] = df["fwd_bytes"] / (df["bwd_bytes"] + 1)
    return df


def safe_numeric(df, cols):
    df = df.copy()
    df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=cols)
    return df


def sample_balanced(df, benign_n, malicious_n, source_name):
    benign = df[df["label"] == 0]
    malicious = df[df["label"] == 1]

    print(f"\n{source_name} distribution before sampling:")
    print(df["label"].value_counts())

    if len(benign) < benign_n:
        raise ValueError(f"{source_name} benign only {len(benign)}, needed {benign_n}")

    if len(malicious) < malicious_n:
        raise ValueError(f"{source_name} malicious only {len(malicious)}, needed {malicious_n}")

    benign_sample = benign.sample(n=benign_n, random_state=RANDOM_STATE)
    malicious_sample = malicious.sample(n=malicious_n, random_state=RANDOM_STATE)

    out = pd.concat([benign_sample, malicious_sample], ignore_index=True)
    out = out.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    out["source"] = source_name

    return out


def save_table(df, name):
    csv_path = TABLE_DIR / f"{name}.csv"
    xlsx_path = TABLE_DIR / f"{name}.xlsx"

    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)

    print(f"Saved CSV : {csv_path}")
    print(f"Saved XLSX: {xlsx_path}")


def compute_metrics(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    if y_score is not None:
        try:
            auc = roc_auc_score(y_true, y_score)
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    return {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": auc,
        "FPR": fpr,
        "FNR": fnr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


def format_metric(x):
    if pd.isna(x):
        return "NaN"
    return f"{x:.4f}"

In [9]:
# ============================================================
# DATASET FILE DETECTION
# RAW UNSW-NB15 + RAW CICIDS2017 WorkingHours
# ============================================================

UNSW_FILES = sorted([
    f for f in glob.glob("./UNSW-NB15_*.csv")
    if re.search(r"UNSW-NB15_[1-4]\.csv$", os.path.basename(f))
])

CICIDS_FILES = sorted([
    f for f in glob.glob("./*.csv")
    if (
        "WorkingHours" in os.path.basename(f)
        or "workingHours" in os.path.basename(f)
    )
])

print("UNSW files used:")
for f in UNSW_FILES:
    print(" -", f)

print("\nCICIDS raw files used:")
for f in CICIDS_FILES:
    print(" -", f)

if len(UNSW_FILES) == 0:
    raise FileNotFoundError("UNSW-NB15_1.csv sampai UNSW-NB15_4.csv tidak ditemukan.")

if len(CICIDS_FILES) == 0:
    raise FileNotFoundError("File CICIDS WorkingHours tidak ditemukan.")

UNSW files used:
 - ./UNSW-NB15_1.csv
 - ./UNSW-NB15_2.csv
 - ./UNSW-NB15_3.csv
 - ./UNSW-NB15_4.csv

CICIDS raw files used:
 - ./Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - ./Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - ./Friday-WorkingHours-Morning.pcap_ISCX.csv
 - ./Monday-WorkingHours.pcap_ISCX.csv
 - ./Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - ./Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - ./Tuesday-WorkingHours.pcap_ISCX.csv
 - ./Wednesday-workingHours.pcap_ISCX.csv


In [10]:
# ============================================================
# FINAL DATASET LOADER SESUAI FOLDER DATA FINAL KAMU
# FILE YANG DIPAKAI:
# 1. UNSW_NB15_merged_all.csv
# 2. CICIDS2017_merged_all.csv
#
# FLOW_DURATION DIHAPUS DARI SEMUA PIPELINE
# SAMPLING: 25.000 BENIGN + 25.000 MALICIOUS PER DATASET
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
BENIGN_PER_DATASET = 25000
MALICIOUS_PER_DATASET = 25000

OUTPUT_DIR = Path("OUTPUT_RESULTS")
OUTPUT_DIR.mkdir(exist_ok=True)

TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

UNSW_FILE = "./UNSW_NB15_merged_all.csv"
CICIDS_FILE = "./CICIDS2017_merged_all.csv"

FEATURE_COLS = [
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
    "total_packets",
    "total_bytes",
    "packet_ratio",
    "byte_ratio"
]

print("DATASET YANG DIPAKAI:")
print("UNSW   :", UNSW_FILE)
print("CICIDS :", CICIDS_FILE)
print("\nFITUR YANG DIPAKAI:")
print(FEATURE_COLS)
print("\nFITUR YANG DIHAPUS: flow_duration")


def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def find_column(df, candidates, target_name):
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"Kolom untuk {target_name} tidak ditemukan.\n"
        f"Kolom tersedia: {list(df.columns)[:100]}"
    )


def save_table(df, name):
    csv_path = TABLE_DIR / f"{name}.csv"
    xlsx_path = TABLE_DIR / f"{name}.xlsx"

    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)

    print("Saved CSV :", csv_path)
    print("Saved XLSX:", xlsx_path)


def prepare_merged_dataset(file_path, source_name):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

    print(f"\nMembaca dataset: {file_path}")
    df = pd.read_csv(file_path, low_memory=False)
    df = clean_column_names(df)

    print(f"\nKolom awal {source_name}:")
    print(list(df.columns)[:50])

    # ========================================================
    # Ambil fitur yang disepakati.
    # flow_duration sengaja TIDAK dipakai.
    # ========================================================

    col_fwd_packets = find_column(
        df,
        [
            "fwd_packets",
            "spkts",
            "total_fwd_packets",
            "tot_fwd_pkts",
            "total_forward_packets"
        ],
        "fwd_packets"
    )

    col_bwd_packets = find_column(
        df,
        [
            "bwd_packets",
            "dpkts",
            "total_backward_packets",
            "tot_bwd_pkts",
            "total_bwd_packets"
        ],
        "bwd_packets"
    )

    col_fwd_bytes = find_column(
        df,
        [
            "fwd_bytes",
            "sbytes",
            "total_length_of_fwd_packets",
            "totlen_fwd_pkts",
            "fwd_packet_length_total",
            "total_len_fwd_packets"
        ],
        "fwd_bytes"
    )

    col_bwd_bytes = find_column(
        df,
        [
            "bwd_bytes",
            "dbytes",
            "total_length_of_bwd_packets",
            "totlen_bwd_pkts",
            "bwd_packet_length_total",
            "total_len_bwd_packets"
        ],
        "bwd_bytes"
    )

    col_label = find_column(
        df,
        [
            "label",
            "label_raw",
            "class",
            "attack_cat"
        ],
        "label"
    )

    out = pd.DataFrame({
        "fwd_packets": df[col_fwd_packets],
        "bwd_packets": df[col_bwd_packets],
        "fwd_bytes": df[col_fwd_bytes],
        "bwd_bytes": df[col_bwd_bytes],
        "label_raw": df[col_label]
    })

    numeric_cols = [
        "fwd_packets",
        "bwd_packets",
        "fwd_bytes",
        "bwd_bytes"
    ]

    out[numeric_cols] = out[numeric_cols].apply(pd.to_numeric, errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=numeric_cols)

    out["label_raw"] = out["label_raw"].astype(str).str.strip().str.upper()

    # ========================================================
    # Mapping label:
    # 0 / BENIGN / NORMAL = benign
    # selain itu = malicious
    # ========================================================

    out["label"] = out["label_raw"].apply(
        lambda x: 0 if x in ["0", "BENIGN", "NORMAL"] else 1
    )

    out = out.drop(columns=["label_raw"])

    # Derived features
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_ratio"] = out["fwd_packets"] / (out["bwd_packets"] + 1)
    out["byte_ratio"] = out["fwd_bytes"] / (out["bwd_bytes"] + 1)

    out = out[FEATURE_COLS + ["label"]]
    out["source"] = source_name

    print(f"\nDistribusi label {source_name} sebelum sampling:")
    print(out["label"].value_counts())

    return out


def stratified_sample_per_dataset(df, source_name, benign_n=25000, malicious_n=25000):
    benign = df[df["label"] == 0]
    malicious = df[df["label"] == 1]

    if len(benign) < benign_n:
        raise ValueError(
            f"{source_name} benign hanya {len(benign)}, "
            f"dibutuhkan {benign_n}."
        )

    if len(malicious) < malicious_n:
        raise ValueError(
            f"{source_name} malicious hanya {len(malicious)}, "
            f"dibutuhkan {malicious_n}."
        )

    benign_sample = benign.sample(
        n=benign_n,
        random_state=RANDOM_STATE
    )

    malicious_sample = malicious.sample(
        n=malicious_n,
        random_state=RANDOM_STATE
    )

    final_df = pd.concat(
        [benign_sample, malicious_sample],
        ignore_index=True
    )

    final_df = final_df.sample(
        frac=1,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    print(f"\nDistribusi label {source_name} setelah sampling:")
    print(final_df["label"].value_counts())

    return final_df


# ============================================================
# 1. BACA FILE MERGED YANG ADA DI FOLDER KAMU
# ============================================================

unsw_all = prepare_merged_dataset(
    UNSW_FILE,
    "UNSW-NB15"
)

cicids_all = prepare_merged_dataset(
    CICIDS_FILE,
    "CICIDS2017"
)


# ============================================================
# 2. SAMPLING SEIMBANG SESUAI REVIEWER
# ============================================================

unsw_final = stratified_sample_per_dataset(
    unsw_all,
    "UNSW-NB15",
    benign_n=BENIGN_PER_DATASET,
    malicious_n=MALICIOUS_PER_DATASET
)

cicids_final = stratified_sample_per_dataset(
    cicids_all,
    "CICIDS2017",
    benign_n=BENIGN_PER_DATASET,
    malicious_n=MALICIOUS_PER_DATASET
)


# ============================================================
# 3. GABUNGKAN MENJADI UNIFIED FLOW DATASET
# ============================================================

unified_df = pd.concat(
    [unsw_final, cicids_final],
    ignore_index=True
)

unified_df = unified_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


# ============================================================
# 4. TABLE 3
# ============================================================

table3 = pd.DataFrame([
    {
        "Dataset": "UNSW-NB15",
        "Records": len(unsw_final),
        "Benign": int((unsw_final["label"] == 0).sum()),
        "Malicious": int((unsw_final["label"] == 1).sum()),
        "Sampling Method": "Stratified balanced sampling"
    },
    {
        "Dataset": "CICIDS2017",
        "Records": len(cicids_final),
        "Benign": int((cicids_final["label"] == 0).sum()),
        "Malicious": int((cicids_final["label"] == 1).sum()),
        "Sampling Method": "Stratified balanced sampling"
    },
    {
        "Dataset": "Unified Flow Dataset",
        "Records": len(unified_df),
        "Benign": int((unified_df["label"] == 0).sum()),
        "Malicious": int((unified_df["label"] == 1).sum()),
        "Sampling Method": "Row-wise concatenation"
    }
])

display(table3)

save_table(table3, "Table_3_Unified_Flow_Dataset")

unified_df.to_csv(
    OUTPUT_DIR / "unified_flow_dataset_balanced_without_flow_duration.csv",
    index=False
)

print("\nSource-label distribution:")
print(unified_df.groupby(["source", "label"]).size())

print("\nFinal class distribution:")
print(unified_df["label"].value_counts())

print("\nDataset final tersimpan di:")
print(OUTPUT_DIR / "unified_flow_dataset_balanced_without_flow_duration.csv")

DATASET YANG DIPAKAI:
UNSW   : ./UNSW_NB15_merged_all.csv
CICIDS : ./CICIDS2017_merged_all.csv

FITUR YANG DIPAKAI:
['fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes', 'total_packets', 'total_bytes', 'packet_ratio', 'byte_ratio']

FITUR YANG DIHAPUS: flow_duration

Membaca dataset: ./UNSW_NB15_merged_all.csv

Kolom awal UNSW-NB15:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime', 'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src__ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'label']

Distribusi label UNSW-NB15 sebelum sampling:
label
1    99643
Name: count, dtype: int64

Membaca datas

ValueError: UNSW-NB15 benign hanya 0, dibutuhkan 25000.

In [ ]:
# ============================================================
# LOAD CICIDS2017 RAW DATASET
# WITHOUT flow_duration
# ============================================================

def load_cicids_balanced_raw(files, benign_n=25000, malicious_n=25000):
    dfs = []

    for file in files:
        print(f"\nLoading CICIDS raw: {file}")

        df = pd.read_csv(file, low_memory=False)
        df = clean_column_names(df)

        col_fwd_packets = find_column(
            df,
            [
                "total_fwd_packets",
                "tot_fwd_pkts",
                "total_forward_packets",
                "fwd_packets"
            ],
            "fwd_packets"
        )

        col_bwd_packets = find_column(
            df,
            [
                "total_backward_packets",
                "tot_bwd_pkts",
                "total_bwd_packets",
                "bwd_packets"
            ],
            "bwd_packets"
        )

        col_fwd_bytes = find_column(
            df,
            [
                "total_length_of_fwd_packets",
                "totlen_fwd_pkts",
                "fwd_packet_length_total",
                "total_len_fwd_packets",
                "fwd_bytes"
            ],
            "fwd_bytes"
        )

        col_bwd_bytes = find_column(
            df,
            [
                "total_length_of_bwd_packets",
                "totlen_bwd_pkts",
                "bwd_packet_length_total",
                "total_len_bwd_packets",
                "bwd_bytes"
            ],
            "bwd_bytes"
        )

        col_label = find_column(df, ["label"], "label")

        temp = pd.DataFrame({
            "fwd_packets": df[col_fwd_packets],
            "bwd_packets": df[col_bwd_packets],
            "fwd_bytes": df[col_fwd_bytes],
            "bwd_bytes": df[col_bwd_bytes],
            "label_raw": df[col_label]
        })

        dfs.append(temp)

    cicids = pd.concat(dfs, ignore_index=True)

    numeric_cols = ["fwd_packets", "bwd_packets", "fwd_bytes", "bwd_bytes"]
    cicids = safe_numeric(cicids, numeric_cols)

    cicids["label_raw"] = cicids["label_raw"].astype(str).str.strip().str.upper()

    print("\nCICIDS raw label distribution:")
    print(cicids["label_raw"].value_counts().head(20))

    # CICIDS label: BENIGN = 0, all attack labels = 1
    cicids["label"] = cicids["label_raw"].apply(
        lambda x: 0 if x == "BENIGN" else 1
    )

    cicids = cicids.drop(columns=["label_raw"])
    cicids = add_derived_features(cicids)

    cicids = cicids[FEATURE_COLS + ["label"]]

    cicids_final = sample_balanced(
        cicids,
        benign_n=benign_n,
        malicious_n=malicious_n,
        source_name="CICIDS2017"
    )

    return cicids_final


cicids_final = load_cicids_balanced_raw(
    CICIDS_FILES,
    benign_n=BENIGN_PER_DATASET,
    malicious_n=MALICIOUS_PER_DATASET
)

print("\nCICIDS final shape:", cicids_final.shape)
print(cicids_final["label"].value_counts())

In [ ]:
# ============================================================
# TABLE 3 - SUMMARY OF CONSTRUCTED UNIFIED FLOW DATASET
# ============================================================

unified_df = pd.concat(
    [unsw_final, cicids_final],
    ignore_index=True
)

unified_df = unified_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

table3 = pd.DataFrame([
    {
        "Dataset": "UNSW-NB15",
        "Records": len(unsw_final),
        "Benign": int((unsw_final["label"] == 0).sum()),
        "Malicious": int((unsw_final["label"] == 1).sum()),
        "Sampling Method": "Stratified balanced sampling"
    },
    {
        "Dataset": "CICIDS2017",
        "Records": len(cicids_final),
        "Benign": int((cicids_final["label"] == 0).sum()),
        "Malicious": int((cicids_final["label"] == 1).sum()),
        "Sampling Method": "Stratified balanced sampling"
    },
    {
        "Dataset": "Unified Flow Dataset",
        "Records": len(unified_df),
        "Benign": int((unified_df["label"] == 0).sum()),
        "Malicious": int((unified_df["label"] == 1).sum()),
        "Sampling Method": "Row-wise concatenation"
    }
])

display(table3)
save_table(table3, "Table_3_Unified_Flow_Dataset")

unified_df.to_csv(
    OUTPUT_DIR / "unified_flow_dataset_balanced_without_flow_duration.csv",
    index=False
)

print("\nSource-label distribution:")
print(unified_df.groupby(["source", "label"]).size())

print("\nFinal class distribution:")
print(unified_df["label"].value_counts())

print("\nSaved unified dataset:")
print(OUTPUT_DIR / "unified_flow_dataset_balanced_without_flow_duration.csv")

In [ ]:
# ============================================================
# FINAL DATASET LOADER SESUAI FOLDER DATA FINAL KAMU
# FILE YANG DIPAKAI:
# 1. UNSW_NB15_merged_all.csv
# 2. CICIDS2017_merged_all.csv
#
# FLOW_DURATION DIHAPUS DARI SEMUA PIPELINE
# SAMPLING: 25.000 BENIGN + 25.000 MALICIOUS PER DATASET
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
BENIGN_PER_DATASET = 25000
MALICIOUS_PER_DATASET = 25000

OUTPUT_DIR = Path("OUTPUT_RESULTS")
OUTPUT_DIR.mkdir(exist_ok=True)

TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(exist_ok=True)

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

UNSW_FILE = "./UNSW_NB15_merged_all.csv"
CICIDS_FILE = "./CICIDS2017_merged_all.csv"

FEATURE_COLS = [
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
    "total_packets",
    "total_bytes",
    "packet_ratio",
    "byte_ratio"
]

print("DATASET YANG DIPAKAI:")
print("UNSW   :", UNSW_FILE)
print("CICIDS :", CICIDS_FILE)
print("\nFITUR YANG DIPAKAI:")
print(FEATURE_COLS)
print("\nFITUR YANG DIHAPUS: flow_duration")


def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def find_column(df, candidates, target_name):
    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"Kolom untuk {target_name} tidak ditemukan.\n"
        f"Kolom tersedia: {list(df.columns)[:100]}"
    )


def save_table(df, name):
    csv_path = TABLE_DIR / f"{name}.csv"
    xlsx_path = TABLE_DIR / f"{name}.xlsx"

    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)

    print("Saved CSV :", csv_path)
    print("Saved XLSX:", xlsx_path)


def prepare_merged_dataset(file_path, source_name):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

    print(f"\nMembaca dataset: {file_path}")
    df = pd.read_csv(file_path, low_memory=False)
    df = clean_column_names(df)

    print(f"\nKolom awal {source_name}:")
    print(list(df.columns)[:50])

    # ========================================================
    # Ambil fitur yang disepakati.
    # flow_duration sengaja TIDAK dipakai.
    # ========================================================

    col_fwd_packets = find_column(
        df,
        [
            "fwd_packets",
            "spkts",
            "total_fwd_packets",
            "tot_fwd_pkts",
            "total_forward_packets"
        ],
        "fwd_packets"
    )

    col_bwd_packets = find_column(
        df,
        [
            "bwd_packets",
            "dpkts",
            "total_backward_packets",
            "tot_bwd_pkts",
            "total_bwd_packets"
        ],
        "bwd_packets"
    )

    col_fwd_bytes = find_column(
        df,
        [
            "fwd_bytes",
            "sbytes",
            "total_length_of_fwd_packets",
            "totlen_fwd_pkts",
            "fwd_packet_length_total",
            "total_len_fwd_packets"
        ],
        "fwd_bytes"
    )

    col_bwd_bytes = find_column(
        df,
        [
            "bwd_bytes",
            "dbytes",
            "total_length_of_bwd_packets",
            "totlen_bwd_pkts",
            "bwd_packet_length_total",
            "total_len_bwd_packets"
        ],
        "bwd_bytes"
    )

    col_label = find_column(
        df,
        [
            "label",
            "label_raw",
            "class",
            "attack_cat"
        ],
        "label"
    )

    out = pd.DataFrame({
        "fwd_packets": df[col_fwd_packets],
        "bwd_packets": df[col_bwd_packets],
        "fwd_bytes": df[col_fwd_bytes],
        "bwd_bytes": df[col_bwd_bytes],
        "label_raw": df[col_label]
    })

    numeric_cols = [
        "fwd_packets",
        "bwd_packets",
        "fwd_bytes",
        "bwd_bytes"
    ]

    out[numeric_cols] = out[numeric_cols].apply(pd.to_numeric, errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=numeric_cols)

    out["label_raw"] = out["label_raw"].astype(str).str.strip().str.upper()

    # ========================================================
    # Mapping label:
    # 0 / BENIGN / NORMAL = benign
    # selain itu = malicious
    # ========================================================

    out["label"] = out["label_raw"].apply(
        lambda x: 0 if x in ["0", "BENIGN", "NORMAL"] else 1
    )

    out = out.drop(columns=["label_raw"])

    # Derived features
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_ratio"] = out["fwd_packets"] / (out["bwd_packets"] + 1)
    out["byte_ratio"] = out["fwd_bytes"] / (out["bwd_bytes"] + 1)

    out = out[FEATURE_COLS + ["label"]]
    out["source"] = source_name

    print(f"\nDistribusi label {source_name} sebelum sampling:")
    print(out["label"].value_counts())

    return out


def stratified_sample_per_dataset(df, source_name, benign_n=25000, malicious_n=25000):
    benign = df[df["label"] == 0]
    malicious = df[df["label"] == 1]

    if len(benign) < benign_n:
        raise ValueError(
            f"{source_name} benign hanya {len(benign)}, "
            f"dibutuhkan {benign_n}."
        )

    if len(malicious) < malicious_n:
        raise ValueError(
            f"{source_name} malicious hanya {len(malicious)}, "
            f"dibutuhkan {malicious_n}."
        )

    benign_sample = benign.sample(
        n=benign_n,
        random_state=RANDOM_STATE
    )

    malicious_sample = malicious.sample(
        n=malicious_n,
        random_state=RANDOM_STATE
    )

    final_df = pd.concat(
        [benign_sample, malicious_sample],
        ignore_index=True
    )

    final_df = final_df.sample(
        frac=1,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    print(f"\nDistribusi label {source_name} setelah sampling:")
    print(final_df["label"].value_counts())

    return final_df


# ============================================================
# 1. BACA FILE MERGED YANG ADA DI FOLDER KAMU
# ============================================================

unsw_all = prepare_merged_dataset(
    UNSW_FILE,
    "UNSW-NB15"
)

cicids_all = prepare_merged_dataset(
    CICIDS_FILE,
    "CICIDS2017"
)


# ============================================================
# 2. SAMPLING SEIMBANG SESUAI REVIEWER
# ============================================================

unsw_final = stratified_sample_per_dataset(
    unsw_all,
    "UNSW-NB15",
    benign_n=BENIGN_PER_DATASET,
    malicious_n=MALICIOUS_PER_DATASET
)

cicids_final = stratified_sample_per_dataset(
    cicids_all,
    "CICIDS2017",
    benign_n=BENIGN_PER_DATASET,
    malicious_n=MALICIOUS_PER_DATASET
)


# ============================================================
# 3. GABUNGKAN MENJADI UNIFIED FLOW DATASET
# ============================================================

unified_df = pd.concat(
    [unsw_final, cicids_final],
    ignore_index=True
)

unified_df = unified_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)


# ============================================================
# 4. TABLE 3
# ============================================================

table3 = pd.DataFrame([
    {
        "Dataset": "UNSW-NB15",
        "Records": len(unsw_final),
        "Benign": int((unsw_final["label"] == 0).sum()),
        "Malicious": int((unsw_final["label"] == 1).sum()),
        "Sampling Method": "Stratified balanced sampling"
    },
    {
        "Dataset": "CICIDS2017",
        "Records": len(cicids_final),
        "Benign": int((cicids_final["label"] == 0).sum()),
        "Malicious": int((cicids_final["label"] == 1).sum()),
        "Sampling Method": "Stratified balanced sampling"
    },
    {
        "Dataset": "Unified Flow Dataset",
        "Records": len(unified_df),
        "Benign": int((unified_df["label"] == 0).sum()),
        "Malicious": int((unified_df["label"] == 1).sum()),
        "Sampling Method": "Row-wise concatenation"
    }
])

display(table3)

save_table(table3, "Table_3_Unified_Flow_Dataset")

unified_df.to_csv(
    OUTPUT_DIR / "unified_flow_dataset_balanced_without_flow_duration.csv",
    index=False
)

print("\nSource-label distribution:")
print(unified_df.groupby(["source", "label"]).size())

print("\nFinal class distribution:")
print(unified_df["label"].value_counts())

print("\nDataset final tersimpan di:")
print(OUTPUT_DIR / "unified_flow_dataset_balanced_without_flow_duration.csv")

NameError: name 'unified_df' is not defined

In [ ]:
# ============================================================
# TABLE 5 - BASELINE MODEL COMPARISON
# ============================================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=None
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

baseline_rows = {}
model_predictions = {}
model_scores = {}

rf_training_time = None

for name, model in models.items():
    print(f"\nTraining {name}...")

    start = time.time()
    model.fit(X_train_pca, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_test_pca)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_pca)[:, 1]
    else:
        y_score = y_pred

    metrics = compute_metrics(y_test, y_pred, y_score)
    baseline_rows[name] = metrics

    model_predictions[name] = y_pred
    model_scores[name] = y_score

    if name == "Random Forest":
        rf_model = model
        rf_pred = y_pred
        rf_score = y_score
        rf_metrics = metrics
        rf_training_time = train_time
        timing["Random Forest training time"] = train_time

# Standalone DBSCAN baseline on test set
print("\nRunning standalone DBSCAN baseline...")

dbscan_baseline = DBSCAN(eps=0.05, min_samples=10)
db_labels = dbscan_baseline.fit_predict(X_test_pca)

# noise = malicious candidate, cluster = benign
db_pred = np.where(db_labels == -1, 1, 0)
db_metrics = compute_metrics(y_test, db_pred, db_pred)

baseline_rows["DBSCAN"] = db_metrics
model_predictions["DBSCAN"] = db_pred
model_scores["DBSCAN"] = db_pred

# Hybrid will be calculated later. Temporary copy RF metrics first.
baseline_rows["Hybrid RF-DBSCAN"] = rf_metrics
model_predictions["Hybrid RF-DBSCAN"] = rf_pred
model_scores["Hybrid RF-DBSCAN"] = rf_score

table5 = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": baseline_rows[name]["Accuracy"],
        "Precision": baseline_rows[name]["Precision"],
        "Recall": baseline_rows[name]["Recall"],
        "F1-score": baseline_rows[name]["F1-score"],
        "ROC-AUC": baseline_rows[name]["ROC-AUC"],
        "FPR": baseline_rows[name]["FPR"],
        "FNR": baseline_rows[name]["FNR"]
    }
    for name in [
        "Logistic Regression",
        "Decision Tree",
        "Gradient Boosting",
        "Random Forest",
        "DBSCAN",
        "Hybrid RF-DBSCAN"
    ]
])

display(table5)
save_table(table5, "Table_5_Baseline_Model_Comparison")

In [ ]:
# ============================================================
# FIGURE 3 - 3D SCIENTIFIC METRIC LANDSCAPE
# ============================================================

def plot_figure3_baseline(table5):
    metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
    model_names = table5["Model"].tolist()

    Z = table5[metric_names].fillna(0).values

    X_grid, Y_grid = np.meshgrid(
        np.arange(len(metric_names)),
        np.arange(len(model_names))
    )

    fig = plt.figure(figsize=(13, 9), facecolor="#f4f8fc")
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(
        X_grid,
        Y_grid,
        Z,
        cmap="viridis",
        alpha=0.85,
        edgecolor="white",
        linewidth=0.8
    )

    ax.plot_wireframe(
        X_grid,
        Y_grid,
        Z,
        color="white",
        linewidth=0.6,
        alpha=0.8
    )

    for i in range(len(model_names)):
        for j in range(len(metric_names)):
            value = Z[i, j]
            ax.scatter(j, i, value, s=55, color="white", edgecolor="black", linewidth=0.5)
            ax.text(
                j,
                i,
                value + 0.035,
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
                color="#111827",
                fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, boxstyle="round,pad=0.15")
            )

    ax.set_title(
        "Figure 3. 3D Scientific Metric Landscape for Baseline Model Comparison",
        fontsize=15,
        fontweight="bold",
        pad=20
    )

    ax.set_xticks(np.arange(len(metric_names)))
    ax.set_xticklabels(metric_names, rotation=20, ha="right")

    ax.set_yticks(np.arange(len(model_names)))
    ax.set_yticklabels(model_names, rotation=-15)

    ax.set_zlabel("Score", labelpad=10)
    ax.set_xlabel("Evaluation Metric", labelpad=10)
    ax.set_ylabel("Model", labelpad=10)
    ax.set_zlim(0, 1.05)

    cbar = fig.colorbar(surf, ax=ax, shrink=0.55, pad=0.12)
    cbar.set_label("Performance Score")

    fig.text(
        0.5,
        0.04,
        "The surface height represents model performance. Higher values indicate stronger detection capability across Accuracy, Precision, Recall, and F1-score.",
        ha="center",
        fontsize=10,
        color="#475569"
    )

    plt.tight_layout()
    path = FIG_DIR / "Figure_3_Baseline_Model_Comparison_3D.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_figure3_baseline(table5)

In [ ]:
# ============================================================
# TRAIN RANDOM FOREST FIRST
# WAJIB DIJALANKAN SEBELUM TABLE 6
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def compute_metrics(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) > 0 else 0,
        "FNR": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    if y_score is not None:
        try:
            metrics["ROC-AUC"] = roc_auc_score(y_true, y_score)
        except Exception:
            metrics["ROC-AUC"] = np.nan
    else:
        metrics["ROC-AUC"] = np.nan

    return metrics


# Pastikan cell split-scaling-PCA sudah dijalankan:
# X_train_pca, X_test_pca, y_train, y_test harus sudah ada

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start = time.time()
rf_model.fit(X_train_pca, y_train)
rf_training_time = time.time() - start

rf_pred = rf_model.predict(X_test_pca)
rf_score = rf_model.predict_proba(X_test_pca)[:, 1]

rf_metrics = compute_metrics(
    y_true=y_test,
    y_pred=rf_pred,
    y_score=rf_score
)

print("Random Forest training selesai.")
print("Training time:", rf_training_time)
print(rf_metrics)

In [ ]:
# ============================================================
# TABLE 6 - RANDOM FOREST CLASSIFICATION PERFORMANCE
# FIGURE 4 - RF CONFUSION MATRIX
# FIGURE 5 - RF FEATURE IMPORTANCE
# ============================================================

table6 = pd.DataFrame([
    {"Metric": "Accuracy", "Value": rf_metrics["Accuracy"]},
    {"Metric": "Precision", "Value": rf_metrics["Precision"]},
    {"Metric": "Recall", "Value": rf_metrics["Recall"]},
    {"Metric": "F1-score", "Value": rf_metrics["F1-score"]},
    {"Metric": "ROC-AUC", "Value": rf_metrics["ROC-AUC"]},
    {"Metric": "False Positive Rate", "Value": rf_metrics["FPR"]},
    {"Metric": "False Negative Rate", "Value": rf_metrics["FNR"]},
    {"Metric": "Training Time (s)", "Value": rf_training_time}
])

display(table6)
save_table(table6, "Table_6_Random_Forest_Classification_Performance")


def plot_confusion_matrix_custom(cm, title, filename, cmap="Blues"):
    tn, fp, fn, tp = cm.ravel()

    fig, ax = plt.subplots(figsize=(7, 6), facecolor="white")

    im = ax.imshow(cm, cmap=cmap)

    labels = [["TN", "FP"], ["FN", "TP"]]
    values = [[tn, fp], [fn, tp]]

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{labels[i][j]}\n{values[i][j]:,}",
                ha="center",
                va="center",
                fontsize=13,
                fontweight="bold",
                color="#111827",
                bbox=dict(
                    facecolor="white",
                    edgecolor="#cbd5e1",
                    alpha=0.95,
                    boxstyle="round,pad=0.25"
                )
            )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Benign", "Malicious"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Benign", "Malicious"])

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title, fontsize=15, fontweight="bold", pad=14)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Number of flows")

    path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


rf_cm = confusion_matrix(y_test, rf_pred, labels=[0, 1])

plot_confusion_matrix_custom(
    rf_cm,
    "Figure 4. Random Forest Confusion Matrix",
    "Figure_4_Random_Forest_Confusion_Matrix.png"
)


# Feature importance based on PCA input is less interpretable.
# Therefore, RF is fitted once on scaled original 8 features for feature importance.
rf_feature_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_feature_model.fit(X_train_scaled, y_train)

feature_importance = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": rf_feature_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(feature_importance)
save_table(feature_importance, "Random_Forest_Feature_Importance")


def plot_feature_importance_heatmap(fi_df):
    df = fi_df.copy()
    features = df["Feature"].tolist()
    importances = df["Importance"].values
    ranks = np.arange(1, len(df) + 1)

    fig, ax = plt.subplots(figsize=(12, 7), facecolor="white")

    y_pos = np.arange(len(features))

    # Left heat strip
    strip = ax.imshow(
        importances.reshape(-1, 1),
        cmap="YlGnBu",
        aspect="auto",
        extent=[0, 0.18, len(features)-0.5, -0.5]
    )

    # Bars
    ax.barh(
        y_pos,
        importances / max(importances) * 0.75,
        left=0.22,
        height=0.62,
        color="#73aef2",
        edgecolor="white"
    )

    for i, (imp, rank) in enumerate(zip(importances, ranks)):
        ax.text(
            0.09,
            i,
            f"{imp:.3f}",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="#111827",
            bbox=dict(facecolor="white", alpha=0.65, edgecolor="none", boxstyle="round,pad=0.15")
        )

        ax.text(
            0.22 + (imp / max(importances) * 0.75) + 0.02,
            i,
            f"rank {rank}",
            ha="left",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="#334155"
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(features)
    ax.invert_yaxis()

    ax.set_xlim(0, 1.15)
    ax.set_xticks([0.09, 0.55, 1.0])
    ax.set_xticklabels(["Importance value", "Medium", "Highest"], fontweight="bold")
    ax.set_xlabel("Relative importance scale", fontweight="bold")
    ax.set_ylabel("Harmonized flow feature", fontweight="bold")

    ax.set_title(
        "Figure 5. Random Forest Feature Importance Heatmap",
        fontsize=15,
        fontweight="bold",
        pad=14
    )

    cbar = fig.colorbar(strip, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("Importance intensity", rotation=270, labelpad=15)

    path = FIG_DIR / "Figure_5_Random_Forest_Feature_Importance_Heatmap.png"
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_feature_importance_heatmap(feature_importance)

NameError: name 'rf_metrics' is not defined

In [ ]:
# ============================================================
# DBSCAN ON LOW-CONFIDENCE FLOWS
# TABLE 7 - DBSCAN PARAMETER TUNING
# FIGURE 6 - DBSCAN CONFUSION MATRIX
# FIGURE 6a - 3D PARAMETER TUNING LANDSCAPE
# ============================================================

# Low-confidence routing threshold
# Ambiguous RF probability range
TAU_L = 0.40
TAU_U = 0.60

low_conf_mask = (rf_score >= TAU_L) & (rf_score <= TAU_U)

X_low = X_test_pca[low_conf_mask]
y_low = np.array(y_test)[low_conf_mask]

print("Low-confidence flows:", len(y_low))
print(pd.Series(y_low).value_counts())

eps_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
min_samples_values = [5, 10, 15, 20]

dbscan_rows = []
best_f1 = -1
best_dbscan_result = None

for eps in eps_values:
    for min_samples in min_samples_values:
        start = time.time()

        if len(y_low) == 0:
            labels = np.array([])
            y_db = np.array([])
            runtime = time.time() - start
            metrics = {k: np.nan for k in ["Accuracy", "Precision", "Recall", "F1-score", "FPR", "FNR"]}
            cluster_count = 0
            noise_count = 0
        else:
            db = DBSCAN(eps=eps, min_samples=min_samples)
            labels = db.fit_predict(X_low)

            # noise = anomaly/malicious candidate, cluster = benign
            y_db = np.where(labels == -1, 1, 0)

            runtime = time.time() - start
            metrics = compute_metrics(y_low, y_db, y_db)

            cluster_count = len(set(labels)) - (1 if -1 in labels else 0)
            noise_count = int((labels == -1).sum())

        row = {
            "eps": eps,
            "min_samples": min_samples,
            "Cluster Count": cluster_count,
            "Noise Count": noise_count,
            "Accuracy": metrics["Accuracy"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1-score": metrics["F1-score"],
            "FPR": metrics["FPR"],
            "FNR": metrics["FNR"],
            "Runtime (s)": runtime
        }

        dbscan_rows.append(row)

        if not pd.isna(metrics["F1-score"]) and metrics["F1-score"] > best_f1:
            best_f1 = metrics["F1-score"]
            best_dbscan_result = {
                "eps": eps,
                "min_samples": min_samples,
                "labels": labels,
                "pred": y_db,
                "metrics": metrics,
                "runtime": runtime
            }

table7 = pd.DataFrame(dbscan_rows)
table7 = table7.sort_values(["F1-score", "Accuracy"], ascending=False).reset_index(drop=True)

display(table7)
save_table(table7, "Table_7_DBSCAN_Parameter_Tuning")

if best_dbscan_result is None:
    raise ValueError("DBSCAN did not produce any valid result. Check low-confidence threshold.")

best_eps = best_dbscan_result["eps"]
best_min_samples = best_dbscan_result["min_samples"]
best_db_pred_low = best_dbscan_result["pred"]
best_db_metrics = best_dbscan_result["metrics"]

print("\nBest DBSCAN parameter:")
print("eps:", best_eps)
print("min_samples:", best_min_samples)
print("F1:", best_db_metrics["F1-score"])


# Figure 6 DBSCAN confusion matrix on low-confidence flows
db_cm = confusion_matrix(y_low, best_db_pred_low, labels=[0, 1])

plot_confusion_matrix_custom(
    db_cm,
    "Figure 6. DBSCAN Confusion Matrix on Low-Confidence Flows",
    "Figure_6_DBSCAN_Confusion_Matrix_Low_Confidence.png"
)


# Figure 6a 3D DBSCAN parameter tuning landscape
def plot_dbscan_3d_landscape(table7):
    fig = plt.figure(figsize=(10, 8), facecolor="#f4f8fc")
    ax = fig.add_subplot(111, projection="3d")

    xs = table7["eps"].values
    ys = table7["min_samples"].values
    zs = table7["F1-score"].fillna(0).values

    scatter = ax.scatter(
        xs,
        ys,
        zs,
        c=zs,
        cmap="viridis",
        s=70,
        edgecolor="black",
        linewidth=0.4
    )

    for x, yv, z in zip(xs, ys, zs):
        ax.text(
            x,
            yv,
            z + 0.025,
            f"{z:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
            color="#111827",
            fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, boxstyle="round,pad=0.12")
        )

    ax.set_title(
        "Figure 6a. 3D DBSCAN Parameter Tuning Landscape",
        fontsize=15,
        fontweight="bold",
        pad=15
    )

    ax.set_xlabel("eps", labelpad=10)
    ax.set_ylabel("min_samples", labelpad=10)
    ax.set_zlabel("F1-score", labelpad=10)
    ax.set_zlim(0, 1.0)

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.55, pad=0.12)
    cbar.set_label("F1-score")

    path = FIG_DIR / "Figure_6a_DBSCAN_Parameter_Tuning_3D.png"
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_dbscan_3d_landscape(table7)

In [ ]:
# ============================================================
# TABLE 8 - HYBRID RF-DBSCAN EVALUATION
# FIGURE 8 - HYBRID RF-DBSCAN CONFUSION MATRIX
# ============================================================

# Build DBSCAN anomaly signal for all test flows.
# Only low-confidence flows receive DBSCAN analysis.
# Non-low-confidence flows get anomaly signal = 0.

start = time.time()

dbscan_signal = np.zeros(len(y_test), dtype=int)

if len(y_low) > 0:
    dbscan_signal[low_conf_mask] = best_db_pred_low

timing["DBSCAN execution time"] = best_dbscan_result["runtime"]

# Validation-based weighting
rf_val_f1 = rf_metrics["F1-score"]
dbscan_val_f1 = best_db_metrics["F1-score"]

if pd.isna(dbscan_val_f1):
    dbscan_val_f1 = 0.0

if (rf_val_f1 + dbscan_val_f1) == 0:
    w_rf = 1.0
    w_dbscan = 0.0
else:
    w_rf = rf_val_f1 / (rf_val_f1 + dbscan_val_f1)
    w_dbscan = dbscan_val_f1 / (rf_val_f1 + dbscan_val_f1)

hybrid_score = (w_rf * rf_score) + (w_dbscan * dbscan_signal)

# Classification threshold
hybrid_pred = np.where(hybrid_score >= 0.5, 1, 0)

timing["Hybrid scoring time"] = time.time() - start

hybrid_metrics = compute_metrics(y_test, hybrid_pred, hybrid_score)

table8 = pd.DataFrame([
    {"Metric": "Accuracy", "Value": hybrid_metrics["Accuracy"]},
    {"Metric": "Precision", "Value": hybrid_metrics["Precision"]},
    {"Metric": "Recall", "Value": hybrid_metrics["Recall"]},
    {"Metric": "F1-score", "Value": hybrid_metrics["F1-score"]},
    {"Metric": "ROC-AUC", "Value": hybrid_metrics["ROC-AUC"]},
    {"Metric": "False Positive Rate", "Value": hybrid_metrics["FPR"]},
    {"Metric": "False Negative Rate", "Value": hybrid_metrics["FNR"]},
    {"Metric": "w_RF", "Value": w_rf},
    {"Metric": "w_DBSCAN", "Value": w_dbscan},
    {"Metric": "Total Execution Time (s)", "Value": timing["DBSCAN execution time"] + timing["Hybrid scoring time"]}
])

display(table8)
save_table(table8, "Table_8_Hybrid_RF_DBSCAN_Evaluation")

# Update Table 5 hybrid row with true hybrid metrics
table5.loc[table5["Model"] == "Hybrid RF-DBSCAN", ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "FPR", "FNR"]] = [
    hybrid_metrics["Accuracy"],
    hybrid_metrics["Precision"],
    hybrid_metrics["Recall"],
    hybrid_metrics["F1-score"],
    hybrid_metrics["ROC-AUC"],
    hybrid_metrics["FPR"],
    hybrid_metrics["FNR"]
]
save_table(table5, "Table_5_Baseline_Model_Comparison_UPDATED")

hybrid_cm = confusion_matrix(y_test, hybrid_pred, labels=[0, 1])

plot_confusion_matrix_custom(
    hybrid_cm,
    "Figure 8. Hybrid RF-DBSCAN Confusion Matrix",
    "Figure_8_Hybrid_RF_DBSCAN_Confusion_Matrix.png"
)

print("Weights:")
print("w_RF:", w_rf)
print("w_DBSCAN:", w_dbscan)

In [ ]:
# ============================================================
# TABLE 9 - ABLATION AND VALIDATION SUMMARY
# FIGURE 9 - ABLATION HEATMAP
# ============================================================

# RF only
rf_only_metrics = rf_metrics

# DBSCAN only, standalone from earlier
dbscan_only_metrics = db_metrics

# Hybrid fixed weight
fixed_w_rf = 0.80
fixed_w_db = 0.20
fixed_score = (fixed_w_rf * rf_score) + (fixed_w_db * dbscan_signal)
fixed_pred = np.where(fixed_score >= 0.5, 1, 0)
fixed_metrics = compute_metrics(y_test, fixed_pred, fixed_score)

# Hybrid validation-based
validation_metrics = hybrid_metrics

# Cross-validation F1 for RF using PCA training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    rf_model,
    X_train_pca,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)
cv_f1 = cv_scores.mean()

table9 = pd.DataFrame([
    {
        "Model": "RF only",
        "Accuracy": rf_only_metrics["Accuracy"],
        "Precision": rf_only_metrics["Precision"],
        "Recall": rf_only_metrics["Recall"],
        "F1-score": rf_only_metrics["F1-score"],
        "ROC-AUC": rf_only_metrics["ROC-AUC"],
        "FPR": rf_only_metrics["FPR"],
        "FNR": rf_only_metrics["FNR"],
        "CV_F1_score": cv_f1
    },
    {
        "Model": "DBSCAN",
        "Accuracy": dbscan_only_metrics["Accuracy"],
        "Precision": dbscan_only_metrics["Precision"],
        "Recall": dbscan_only_metrics["Recall"],
        "F1-score": dbscan_only_metrics["F1-score"],
        "ROC-AUC": dbscan_only_metrics["ROC-AUC"],
        "FPR": dbscan_only_metrics["FPR"],
        "FNR": dbscan_only_metrics["FNR"],
        "CV_F1_score": np.nan
    },
    {
        "Model": "Hybrid Fixed Weight",
        "Accuracy": fixed_metrics["Accuracy"],
        "Precision": fixed_metrics["Precision"],
        "Recall": fixed_metrics["Recall"],
        "F1-score": fixed_metrics["F1-score"],
        "ROC-AUC": fixed_metrics["ROC-AUC"],
        "FPR": fixed_metrics["FPR"],
        "FNR": fixed_metrics["FNR"],
        "CV_F1_score": cv_f1
    },
    {
        "Model": "Hybrid Validation-Based Weight",
        "Accuracy": validation_metrics["Accuracy"],
        "Precision": validation_metrics["Precision"],
        "Recall": validation_metrics["Recall"],
        "F1-score": validation_metrics["F1-score"],
        "ROC-AUC": validation_metrics["ROC-AUC"],
        "FPR": validation_metrics["FPR"],
        "FNR": validation_metrics["FNR"],
        "CV_F1_score": cv_f1
    }
])

display(table9)
save_table(table9, "Table_9_Ablation_and_Validation_Summary")


def plot_ablation_heatmap(table9):
    metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
    data = table9.set_index("Model")[metrics].fillna(0)

    fig, ax = plt.subplots(figsize=(11, 6), facecolor="white")

    im = ax.imshow(data.values, cmap="coolwarm", vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(metrics)))
    ax.set_xticklabels(metrics, fontweight="bold")

    ax.set_yticks(np.arange(len(data.index)))
    ax.set_yticklabels(data.index)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data.values[i, j]
            text_color = "white" if value > 0.65 else "#111827"
            ax.text(
                j,
                i,
                f"{value:.3f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
                fontweight="bold",
                bbox=dict(facecolor="black" if value > 0.65 else "white",
                          alpha=0.25 if value > 0.65 else 0.65,
                          edgecolor="none",
                          boxstyle="round,pad=0.15")
            )

    ax.set_title(
        "Figure 9. Ablation and Validation Metric Heatmap",
        fontsize=15,
        fontweight="bold",
        pad=14
    )

    ax.set_xlabel("Evaluation Metric", fontweight="bold")
    ax.set_ylabel("Model", fontweight="bold")

    cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("Performance score", rotation=270, labelpad=15)

    path = FIG_DIR / "Figure_9_Ablation_and_Validation_Heatmap.png"
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_ablation_heatmap(table9)

In [ ]:
# ============================================================
# TABLE 10 - IP REPUTATION CATEGORY DISTRIBUTION
# FIGURE 10 - HYBRID IP REPUTATION NETWORK DISTRIBUTION
# FIGURE 11 - 3D HYBRID REPUTATION SCORE SPACE
# ============================================================

def map_reputation(score):
    if score < 0.30:
        return "Whitelist"
    elif score < 0.70:
        return "Graylist"
    else:
        return "Blacklist"

reputation_category = pd.Series(hybrid_score).apply(map_reputation)

table10 = (
    reputation_category
    .value_counts()
    .reindex(["Whitelist", "Graylist", "Blacklist"])
    .fillna(0)
    .astype(int)
    .reset_index()
)

table10.columns = ["Reputation Category", "Count"]
table10["Percentage"] = table10["Count"] / table10["Count"].sum() * 100

interpretation_map = {
    "Whitelist": "Low-risk traffic",
    "Graylist": "Uncertain traffic requiring monitoring",
    "Blacklist": "High-risk traffic"
}

table10["Interpretation"] = table10["Reputation Category"].map(interpretation_map)

display(table10)
save_table(table10, "Table_10_IP_Reputation_Category_Distribution")


def plot_reputation_network_distribution(table10):
    counts = dict(zip(table10["Reputation Category"], table10["Count"]))
    percentages = dict(zip(table10["Reputation Category"], table10["Percentage"]))

    whitelist = counts.get("Whitelist", 0)
    graylist = counts.get("Graylist", 0)
    blacklist = counts.get("Blacklist", 0)

    wp = percentages.get("Whitelist", 0)
    gp = percentages.get("Graylist", 0)
    bp = percentages.get("Blacklist", 0)

    fig, ax = plt.subplots(figsize=(14, 8), facecolor="#f5f9ff")
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 8)
    ax.axis("off")

    # Background random network dots
    rng = np.random.default_rng(RANDOM_STATE)
    for _ in range(90):
        x = rng.uniform(0.5, 13.5)
        yv = rng.uniform(0.8, 7.2)
        ax.scatter(x, yv, s=10, color="#94a3b8", alpha=0.35)
        if rng.random() < 0.45:
            x2 = x + rng.uniform(-0.8, 0.8)
            y2 = yv + rng.uniform(-0.8, 0.8)
            ax.plot([x, x2], [yv, y2], color="#cbd5e1", alpha=0.25, linewidth=0.8)

    ax.text(
        7,
        7.55,
        "Figure 10. Hybrid IP Reputation Network Distribution",
        ha="center",
        fontsize=18,
        fontweight="bold",
        color="#0f172a"
    )

    ax.text(
        7,
        7.15,
        "Firewall-oriented risk mapping based on the hybrid RF–DBSCAN reputation score",
        ha="center",
        fontsize=11,
        color="#475569"
    )

    # Threshold box
    ax.text(
        7,
        6.35,
        "Reputation Thresholds\n\nScore < 0.3 → Whitelist     0.3 ≤ Score < 0.7 → Graylist     Score ≥ 0.7 → Blacklist",
        ha="center",
        va="center",
        fontsize=10,
        color="#334155",
        bbox=dict(facecolor="white", edgecolor="#bfdbfe", boxstyle="round,pad=0.7")
    )

    # Center hybrid score circle
    center = plt.Circle((7, 4.05), 0.75, fill=False, linewidth=3, color="#2563eb")
    ax.add_patch(center)

    ax.text(7, 4.25, "Hybrid\nScore", ha="center", va="center", fontsize=14, fontweight="bold", color="#0f172a")
    ax.text(7, 3.65, "RF probability\nDBSCAN signal", ha="center", va="center", fontsize=8, color="#475569")

    # Category circles
    categories = [
        ("Whitelist", whitelist, wp, 2.6, 4.0, "#16a34a", "Low-risk traffic"),
        ("Graylist", graylist, gp, 7.0, 1.75, "#f59e0b", "Uncertain traffic requiring monitoring"),
        ("Blacklist", blacklist, bp, 11.4, 4.0, "#dc2626", "High-risk traffic")
    ]

    for name, count, pct, x, yv, color, desc in categories:
        # aura rings
        for r in [0.95, 1.15, 1.35, 1.55, 1.75]:
            ax.add_patch(plt.Circle((x, yv), r, fill=False, color=color, alpha=0.12, linewidth=1))

        ax.add_patch(plt.Circle((x, yv), 1.25, fill=False, color=color, linewidth=3))

        ax.text(x, yv + 0.35, name, ha="center", va="center", fontsize=14, fontweight="bold", color=color)
        ax.text(x, yv - 0.02, f"{count:,}", ha="center", va="center", fontsize=16, fontweight="bold", color="#111827")
        ax.text(x, yv - 0.45, f"{pct:.2f}%", ha="center", va="center", fontsize=10, color="#475569")
        ax.text(x, yv - 1.35, desc, ha="center", va="center", fontsize=9, color="#475569")

    # Arrows
    ax.annotate("", xy=(3.85, 4.0), xytext=(6.25, 4.0), arrowprops=dict(arrowstyle="->", color="#16a34a", lw=2.2))
    ax.annotate("", xy=(7, 2.95), xytext=(7, 3.3), arrowprops=dict(arrowstyle="->", color="#f59e0b", lw=2.2))
    ax.annotate("", xy=(10.15, 4.0), xytext=(7.75, 4.0), arrowprops=dict(arrowstyle="->", color="#dc2626", lw=2.2))

    ax.text(
        7,
        0.35,
        "Node size represents the proportion of flows assigned to each reputation category.",
        ha="center",
        fontsize=10,
        color="#475569"
    )

    path = FIG_DIR / "Figure_10_Hybrid_IP_Reputation_Network_Distribution.png"
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_reputation_network_distribution(table10)


def plot_3d_hybrid_reputation_space():
    sample_n = min(1500, len(hybrid_score))
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(np.arange(len(hybrid_score)), size=sample_n, replace=False)

    x = rf_score[idx]
    yv = hybrid_score[idx]
    z = dbscan_signal[idx]
    c = np.array(y_test)[idx]

    fig = plt.figure(figsize=(12, 8), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    sc = ax.scatter(
        x,
        yv,
        z,
        c=c,
        cmap="coolwarm",
        s=18,
        alpha=0.75,
        edgecolor="white",
        linewidth=0.2
    )

    # threshold planes
    yy, zz = np.meshgrid(np.linspace(0, 1, 2), np.linspace(0, 1, 2))
    xx1 = np.full_like(yy, 0.3)
    xx2 = np.full_like(yy, 0.7)

    ax.plot_surface(xx1, yy, zz, alpha=0.10, color="green")
    ax.plot_surface(xx2, yy, zz, alpha=0.10, color="red")

    ax.text(0.03, 0.05, 1.02, "Whitelist threshold = 0.3", color="#166534", fontweight="bold")
    ax.text(0.45, 0.05, 1.02, "Blacklist threshold = 0.7", color="#991b1b", fontweight="bold")

    ax.set_xlabel("Random Forest Malicious Probability", labelpad=10)
    ax.set_ylabel("Hybrid Reputation Score", labelpad=10)
    ax.set_zlabel("DBSCAN Anomaly Signal", labelpad=10)

    ax.set_title(
        "Figure 11. 3D Hybrid Reputation Score Space",
        fontsize=15,
        fontweight="bold",
        pad=16
    )

    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.12)
    cbar.set_label("True class")
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Benign", "Malicious"])

    path = FIG_DIR / "Figure_11_3D_Hybrid_Reputation_Score_Space.png"
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_3d_hybrid_reputation_space()

In [ ]:
# ============================================================
# TABLE 11 - COMPUTATIONAL COST AND ERROR SUMMARY
# FIGURE 12 - COMPUTATIONAL COST AND ERROR ANALYSIS DASHBOARD
# ============================================================

tn, fp, fn, tp = hybrid_cm.ravel()

graylist_mask = reputation_category == "Graylist"
graylist_flows = int(graylist_mask.sum())

y_test_array = np.array(y_test)

graylist_benign = int(((graylist_mask.values) & (y_test_array == 0)).sum())
graylist_malicious = int(((graylist_mask.values) & (y_test_array == 1)).sum())

table11 = pd.DataFrame([
    {"Aspect": "Train-test split time", "Value": timing.get("Train-test split time", np.nan), "Interpretation": "Dataset partitioning cost"},
    {"Aspect": "Min-Max Scaling time", "Value": timing.get("Min-Max Scaling time", np.nan), "Interpretation": "Feature normalization cost"},
    {"Aspect": "PCA transformation time", "Value": timing.get("PCA transformation time", np.nan), "Interpretation": "Dimensionality reduction cost"},
    {"Aspect": "Random Forest training time", "Value": timing.get("Random Forest training time", np.nan), "Interpretation": "Supervised model training cost"},
    {"Aspect": "DBSCAN execution time", "Value": timing.get("DBSCAN execution time", np.nan), "Interpretation": "Low-confidence anomaly analysis cost"},
    {"Aspect": "Hybrid scoring time", "Value": timing.get("Hybrid scoring time", np.nan), "Interpretation": "Reputation score computation cost"},
    {"Aspect": "False Positive", "Value": int(fp), "Interpretation": "Benign flows classified as malicious"},
    {"Aspect": "False Negative", "Value": int(fn), "Interpretation": "Malicious flows classified as benign"},
    {"Aspect": "Graylist flows", "Value": graylist_flows, "Interpretation": "Flows requiring further monitoring"},
    {"Aspect": "Graylist benign flows", "Value": graylist_benign, "Interpretation": "Benign traffic placed under monitoring"},
    {"Aspect": "Graylist malicious flows", "Value": graylist_malicious, "Interpretation": "Malicious traffic requiring monitoring"}
])

display(table11)
save_table(table11, "Table_11_Computational_Cost_and_Error_Summary")


def plot_computational_cost_dashboard(table11):
    cost_aspects = [
        "Train-test split time",
        "Min-Max Scaling time",
        "PCA transformation time",
        "Random Forest training time",
        "DBSCAN execution time",
        "Hybrid scoring time"
    ]

    error_aspects = [
        "False Positive",
        "False Negative",
        "Graylist flows",
        "Graylist benign flows",
        "Graylist malicious flows"
    ]

    cost_df = table11[table11["Aspect"].isin(cost_aspects)].copy()
    error_df = table11[table11["Aspect"].isin(error_aspects)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor="white")

    fig.suptitle(
        "Figure 12. Computational Cost and Error Analysis Dashboard",
        fontsize=18,
        fontweight="bold",
        color="#0f172a",
        y=0.98
    )

    # Panel A
    ax = axes[0]
    ax.set_title("A. Computational Cost", fontsize=14, fontweight="bold", pad=12)
    ax.set_facecolor("#f8fafc")

    y_pos = np.arange(len(cost_df))
    max_cost = max(cost_df["Value"].astype(float).max(), 1e-9)

    ax.barh(
        y_pos,
        [max_cost] * len(cost_df),
        color="#e5e7eb",
        height=0.58
    )

    ax.barh(
        y_pos,
        cost_df["Value"].astype(float),
        color="#38bdf8",
        height=0.58
    )

    for i, v in enumerate(cost_df["Value"].astype(float)):
        ax.text(
            min(v + max_cost * 0.03, max_cost * 0.96),
            i,
            f"{v:.4f} s",
            va="center",
            ha="left",
            fontsize=10,
            fontweight="bold",
            color="#111827",
            bbox=dict(facecolor="white", edgecolor="#cbd5e1", boxstyle="round,pad=0.25")
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(cost_df["Aspect"])
    ax.invert_yaxis()
    ax.set_xlim(0, max_cost * 1.25)
    ax.set_xticks([])
    ax.spines[:].set_visible(False)

    ax.text(
        0,
        len(cost_df) + 0.3,
        "Values are shown in seconds.",
        fontsize=9,
        color="#475569"
    )

    # Panel B
    ax2 = axes[1]
    ax2.set_title("B. Error and Monitoring Analysis", fontsize=14, fontweight="bold", pad=12)
    ax2.set_facecolor("#f8fafc")

    y_pos2 = np.arange(len(error_df))
    max_error = max(error_df["Value"].astype(float).max(), 1)

    bg_color = "#e5e7eb"
    colors = ["#ef4444", "#f97316", "#f59e0b", "#22c55e", "#8b5cf6"]

    ax2.barh(
        y_pos2,
        [max_error] * len(error_df),
        color=bg_color,
        height=0.58
    )

    ax2.barh(
        y_pos2,
        error_df["Value"].astype(float),
        color=colors[:len(error_df)],
        height=0.58
    )

    for i, v in enumerate(error_df["Value"].astype(int)):
        ax2.text(
            min(v + max_error * 0.03, max_error * 0.96),
            i,
            f"{v:,}",
            va="center",
            ha="left",
            fontsize=10,
            fontweight="bold",
            color="#111827",
            bbox=dict(facecolor="white", edgecolor="#cbd5e1", boxstyle="round,pad=0.25")
        )

    ax2.set_yticks(y_pos2)
    ax2.set_yticklabels(error_df["Aspect"])
    ax2.invert_yaxis()
    ax2.set_xlim(0, max_error * 1.25)
    ax2.set_xticks([])
    ax2.spines[:].set_visible(False)

    ax2.text(
        0,
        len(error_df) + 0.3,
        "Values are shown as flow counts.",
        fontsize=9,
        color="#475569"
    )

    fig.text(
        0.5,
        0.03,
        "Computational time and error counts are separated because they have different numerical scales.",
        ha="center",
        fontsize=10,
        color="#475569"
    )

    path = FIG_DIR / "Figure_12_Computational_Cost_and_Error_Analysis_Dashboard.png"
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


plot_computational_cost_dashboard(table11)

In [ ]:
# ============================================================
# FINAL SUMMARY FOR PAPER
# ============================================================

print("\n================ FINAL EXPERIMENT SUMMARY ================")

print("\nDataset used:")
print("- Raw UNSW-NB15 files: UNSW-NB15_1.csv to UNSW-NB15_4.csv")
print("- Raw CICIDS2017 WorkingHours CSV files")
print("- Excluded feature: flow_duration")
print("- Final features:", FEATURE_COLS)

print("\nTable 3:")
display(table3)

print("\nTable 5:")
display(table5)

print("\nTable 6:")
display(table6)

print("\nTable 7:")
display(table7.head(10))

print("\nTable 8:")
display(table8)

print("\nTable 9:")
display(table9)

print("\nTable 10:")
display(table10)

print("\nTable 11:")
display(table11)

print("\nAll outputs saved in:")
print(OUTPUT_DIR.resolve())